# Part 2: Chunking Lab

Chunk the corpus with five strategies, index each in Qdrant and compare them.

In [1]:
from prime_land.config import CONFIG, get_path
from prime_land.ingest_document.chunkers import chunk_documents
from prime_land.ingest_document.comparison import compare_strategies
from prime_land.ingest_document.indexer import collection_name, get_client, index_chunks, save_chunks
from prime_land.ingest_document.writers import load_corpus
from prime_land.providers import get_embeddings

documents = load_corpus(get_path("corpus_file"), CONFIG["corpus"]["exclude_patterns"])
strategies = CONFIG["chunking"]["strategies"]
print(len(documents), "documents")

77 documents


In [2]:
chunk_sets = {}
for strategy in strategies:
    chunk_sets[strategy] = chunk_documents(documents, strategy)
    save_chunks(chunk_sets[strategy], strategy)
    print(strategy, len(chunk_sets[strategy]), "chunks")

semantic 455 chunks
fixed 185 chunks
sliding 363 chunks
parent_child 580 chunks
late 148 chunks


In [3]:
embeddings = get_embeddings()
try:
    client.close()
except NameError:
    pass
client = get_client()
stores = {}
budget = 900
for strategy in strategies:
    name = collection_name(strategy)
    before = client.count(name).count if client.collection_exists(name) else 0
    if budget <= 0:
        break
    stores[strategy] = index_chunks(client, embeddings, chunk_sets[strategy], strategy, limit=budget)
    budget -= client.count(name).count - before
    print(strategy, client.count(name).count, 'of', len(chunk_sets[strategy]), 'budget left', budget)


semantic: resuming from 455 of 455
semantic 455 of 455 budget left 900
fixed: resuming from 185 of 185
fixed 185 of 185 budget left 900
sliding: resuming from 363 of 363
sliding 363 of 363 budget left 900
parent_child: resuming from 580 of 580
parent_child 580 of 580 budget left 900
late: resuming from 148 of 148
late 148 of 148 budget left 900


In [4]:
table = compare_strategies(client, embeddings, stores, chunk_sets)
table

rate limited, waiting 60s
token budget used 17226, waiting 40s
token budget used 17633, waiting 0s
token budget used 17625, waiting 1s
token budget used 17634, waiting 39s


,strategy,chunk_count,avg_tokens,index_size_mb,avg_retrieval_ms
0,semantic,455,233.3,1.40,612.0
1,fixed,185,607.7,0.57,630.9
2,sliding,363,485.7,1.12,672.7
3,parent_child,580,205.1,1.78,605.1
4,late,148,696.3,0.45,20004.4


In [5]:
client.close()